# 4 · ESM-IF (esm_if1) - sequence design
`esm.inverse_folding`. The original notebook's install broke on Colab; this one
pins compatible torch-geometric wheels. Uses the shared v6 structures and
temperature-only sampling. **Runtime → GPU.**

In [ ]:
#@title Step 0 - Upload & unzip the design bundle
#@markdown Upload **design_bundle.zip** (contains `design_common.py`,
#@markdown `design_input_proteins.csv`, and `structures/`).
#@markdown Build it locally with `design/make_bundle.sh`.
import os, zipfile
from google.colab import files

if not os.path.exists("design_common.py"):
    print("Upload design_bundle.zip:")
    up = files.upload()
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall(".")
    # if it unzipped into a 'design/' subdir, hoist contents to CWD
    if os.path.exists("design/design_common.py") and not os.path.exists("design_common.py"):
        import shutil
        for item in os.listdir("design"):
            shutil.move(os.path.join("design", item), item)
print("Bundle ready:", sorted(os.listdir(".")))

In [ ]:
#@title Install ESM-IF deps  (fair-esm + torch_geometric + compiled companions)
import torch, subprocess, sys
tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".","") if torch.version.cuda else "cpu"
print("torch", tv, cu)
# (1) ESM-IF needs fair-esm to OWN the `esm` import namespace. If EvolutionaryScale's
#     `esm` (ESM3) package is present it shadows fair-esm and `esm.pretrained`
#     disappears -> remove it first (ESM3 belongs in its own notebook/runtime).
subprocess.run([sys.executable,"-m","pip","uninstall","-y","esm"], check=False)
# (2) torch_geometric is the piece most installs forget (fair-esm imports it directly).
subprocess.run([sys.executable,"-m","pip","install","--quiet","torch_geometric"], check=True)
# (3) compiled companions, matched to the runtime's torch+CUDA build
subprocess.run([sys.executable,"-m","pip","install","--quiet",
    "torch-scatter","torch-sparse","torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"], check=False)
subprocess.run([sys.executable,"-m","pip","install","--quiet","fair-esm"], check=True)
# (4) fair-esm 2.0.1 uses biotite's OLD API (`filter_backbone`), removed in biotite >=1.0.
#     Pin biotite<1.0 or `import esm.inverse_folding` fails with an ImportError.
subprocess.run([sys.executable,"-m","pip","install","--quiet","biotite==0.41.1"], check=True)
# sanity: fair-esm must expose pretrained + the IF loader must import cleanly
import importlib, esm; importlib.reload(esm)
import esm.inverse_folding  # forces the biotite-dependent import to run now
assert hasattr(esm, "pretrained"), "esm.pretrained missing - ESM3 still shadowing fair-esm? Restart runtime."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set GPU runtime!)")

In [ ]:
#@title Step 1 - Import shared config and show the LOCKED settings
import design_common as dc
proteins = dc.load_inputs()           # 25 templates, structure paths resolved
print(f"Loaded {len(proteins)} design templates")
print("\n=== LOCKED CONFIG (identical across all model notebooks) ===")
import dataclasses, json
cfg = {k: v for k, v in dataclasses.asdict(dc.CONFIG).items() if k != "deviations"}
print(json.dumps(cfg, indent=2, default=str))
display(proteins[["uniprot_id","species","domain","rank_class","sequence_length"]])

In [ ]:
#@title Load ESM-IF  (correct loader name = esm_if1_gvp4_t16_142M_UR50)
import torch, esm, esm.inverse_folding as esmif
MODEL = "ESM-IF"; SOLUBLE = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esmif_model, esm_alphabet = esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
esmif_model = esmif_model.eval().to(device)
print("Loaded ESM-IF on", device)

## Comparability notes - ESM-IF

These are the points where ESM-IF touches the locked settings. Anything that
**deviates** is recorded via `dc.CONFIG.note_deviation(...)` so it lands in the
output manifest.


- **Structure**: shared v6 PDB, chain A (original notebook downloaded v4 - fixed).
- **Sampler**: `model.sample(coords, temperature=...)` is temperature-only - matches.
- **score_type** = `mean_logp` from `esmif.util.score_sequence` (per-residue conditional log-likelihood of the design given the backbone).
- ESM-IF was trained partly on AlphaFold structures - note the circularity caveat in the paper.

In [ ]:
#@title ESM-IF helpers
import torch, esm.inverse_folding as esmif
def load_coords(pdb_path):
    structure = esmif.util.load_structure(pdb_path, dc.CONFIG.design_chain)
    coords, _ = esmif.util.extract_coords_from_structure(structure)
    return coords
@torch.no_grad()
def design_one(coords, seed):
    torch.manual_seed(seed)
    # NOTE: GVPTransformerModel.sample() signature is (coords, partial_seq, temperature, confidence)
    # - there is no `device` kwarg; it uses the model's device.
    seq = esmif_model.sample(coords, temperature=dc.CONFIG.temperature)
    # conditional log-likelihood of the sampled sequence given the backbone
    ll, _ = esmif.util.score_sequence(esmif_model, esm_alphabet, coords, seq)
    return seq, float(ll)

In [ ]:
#@title Step 2 - Smoke test (shortest protein, one seed)
_p = proteins.sort_values("sequence_length").iloc[0]
_coords = load_coords(_p.structure_path)
_seq, _ll = design_one(_coords, seed=dc.CONFIG.seeds[0])
print(f"{_p.uniprot_id} len={_p.sequence_length}  mean_logp={_ll:.3f}")
print("DES:", _seq[:60])
assert len(_seq) == _p.sequence_length and set(_seq) <= set(dc.CANONICAL_AA)
print("✓ smoke test OK")

In [ ]:
#@title Step 3 - Design all 25 proteins
from tqdm.auto import tqdm
rows = []
for p in tqdm(list(proteins.itertuples()), desc="ESM-IF design"):
    coords = load_coords(p.structure_path)
    for i, seed in enumerate(dc.CONFIG.seeds):
        seq, ll = design_one(coords, seed=seed)
        rows.append(dc.make_record(p, model=MODEL, sample_idx=i, seed=seed,
                                   designed_sequence=seq, model_score=ll,
                                   score_type="mean_logp", soluble_variant=SOLUBLE))
print(f"Generated {len(rows)} sequences")

In [ ]:
#@title Step 4 - Validate (faithful + comparable) and write outputs
df = dc.finalize(rows, model=MODEL, strict=True)   # raises if a guard fails
dc.write_designs(df, MODEL)
dc.write_fasta(df, MODEL)

# Quick faithfulness readout: per-protein WT sequence recovery distribution
import numpy as np
rec = df.apply(lambda r: sum(a==b for a,b in zip(r.designed_sequence, r.wt_sequence))/r.seq_length, axis=1)
print(f"\nSeq-recovery vs WT - median {rec.median():.1%}, "
      f"IQR [{rec.quantile(.25):.1%}, {rec.quantile(.75):.1%}]")
print("(Inverse-folding designs typically recover ~30-55% of WT; "
      "near-100% means the sampler is too cold / stuck, near-5% means random.)")

from google.colab import files
files.download(str(dc.OUTPUT_DIR / f"designs_{MODEL.replace('/','_').replace('-','-')}.csv"))